# 📊 SEC EDGAR — SQL Analysis
**Project:** SEC EDGAR Financial Ratio Analysis  
**Engineer:** Meet Saini  
**Last Updated:** 2026-05-31

---

### Business Question
"Which financial KPIs derived from SEC EDGAR filings were most impacted 
by COVID-19, how did they recover Post-COVID, and do these patterns 
vary by industry sector?"

### Input Table
`edgar_kpi_unified`

| Column | Type | Description |
|--------|------|-------------|
| `adsh` | string | Accession number — unique filing identifier |
| `cik` | string | Central Index Key — unique company identifier |
| `name` | string | Company name |
| `sic` | integer | Standard Industrial Classification — industry code |
| `filed` | date | Date filing was submitted to SEC |
| `source_year` | integer | Year of filing |
| `source_quarter` | integer | Quarter of filing |
| `period` | string | COVID period — Pre-COVID, COVID-Impact, Post-COVID-Recovery |
| `current_ratio` | double | Current Assets / Current Liabilities |
| `debt_ratio` | double | Total Liabilities / Total Assets |
| `roa` | double | Net Income / Total Assets |
| `roe` | double | Net Income / Stockholders Equity |
| `operating_margin` | double | Operating Income / Revenue |
| `gross_margin` | double | Gross Profit / Revenue |
| `asset_turnover` | double | Revenue / Total Assets |
| `debt_to_equity` | double | Long Term Debt / Stockholders Equity |

### KPI Coverage
| KPI | Rows | Coverage |
|-----|------|----------|
| current_ratio | 150,347 | High ✅ |
| debt_ratio | 160,891 | High ✅ |
| roa | 52,401 | Medium ✅ |
| roe | 49,531 | Medium ✅ |
| operating_margin | 29,082 | Medium ✅ |
| gross_margin | 16,793 | Low-Medium ⚠️ |
| asset_turnover | 35,181 | Medium ✅ |
| debt_to_equity | 9,662 | Low ⚠️ |

## Q1 — How did each KPI trend across periods?
Compares average value of all 8 KPIs across Pre-COVID, COVID-Impact
and Post-COVID-Recovery periods.
Establishes the macro-level impact of COVID on financial health.

**Expected outcome:** KPIs like ROA, ROE and operating margin should 
drop during COVID-Impact and recover Post-COVID. Leverage ratios 
like debt ratio may increase during COVID as companies took on more debt.

In [0]:
%sql
SELECT
    period,
    COUNT(DISTINCT cik)                    AS companies,
    ROUND(AVG(current_ratio), 4)           AS avg_current_ratio,
    ROUND(AVG(debt_ratio), 4)              AS avg_debt_ratio,
    ROUND(AVG(roa), 4)                     AS avg_roa,
    ROUND(AVG(roe), 4)                     AS avg_roe,
    ROUND(AVG(operating_margin), 4)        AS avg_operating_margin,
    ROUND(AVG(gross_margin), 4)            AS avg_gross_margin,
    ROUND(AVG(asset_turnover), 4)          AS avg_asset_turnover,
    ROUND(AVG(debt_to_equity), 4)          AS avg_debt_to_equity
FROM edgar_kpi_clean
GROUP BY period
ORDER BY
    CASE period
        WHEN 'Pre-COVID' THEN 1
        WHEN 'COVID-Impact' THEN 2
        WHEN 'Post-COVID-Recovery' THEN 3
    END

## Q2 — Which sectors were most financially impacted during COVID?
Compares average KPIs by SIC sector during COVID-Impact period
against Pre-COVID baseline.
Sectors with largest negative delta were most impacted.

**Expected outcome:** Hospitality, retail and energy sectors 
should show the largest drops in ROA and operating margin 
during COVID-Impact.

In [0]:
%sql
WITH pre_covid AS (
    SELECT sic,
        COUNT(DISTINCT cik)       AS companies,
        AVG(roa)                  AS avg_roa,
        AVG(operating_margin)     AS avg_operating_margin,
        AVG(current_ratio)        AS avg_current_ratio
    FROM edgar_kpi_clean
    WHERE period = 'Pre-COVID'
    AND sic IS NOT NULL
    GROUP BY sic
    HAVING COUNT(DISTINCT cik) >= 10
),
covid_impact AS (
    SELECT sic,
        AVG(roa)                  AS avg_roa,
        AVG(operating_margin)     AS avg_operating_margin,
        AVG(current_ratio)        AS avg_current_ratio
    FROM edgar_kpi_clean
    WHERE period = 'COVID-Impact'
    AND sic IS NOT NULL
    GROUP BY sic
)
SELECT
    p.sic,
    p.companies,
    ROUND(p.avg_roa, 4)                                            AS pre_covid_roa,
    ROUND(c.avg_roa, 4)                                            AS covid_roa,
    ROUND(c.avg_roa - p.avg_roa, 4)                               AS roa_delta,
    ROUND(p.avg_operating_margin, 4)                               AS pre_covid_op_margin,
    ROUND(c.avg_operating_margin, 4)                               AS covid_op_margin,
    ROUND(c.avg_operating_margin - p.avg_operating_margin, 4)     AS op_margin_delta,
    ROUND(p.avg_current_ratio, 4)                                  AS pre_covid_current_ratio,
    ROUND(c.avg_current_ratio, 4)                                  AS covid_current_ratio,
    ROUND(c.avg_current_ratio - p.avg_current_ratio, 4)           AS current_ratio_delta
FROM pre_covid p
JOIN covid_impact c ON p.sic = c.sic
ORDER BY roa_delta ASC
LIMIT 20

## Q3 — Which KPIs took longest to recover Post-COVID?
Compares all 8 KPIs across all 3 periods side by side.
A KPI has recovered if Post-COVID value >= Pre-COVID baseline.
A KPI has not recovered if Post-COVID value < Pre-COVID baseline.

**Expected outcome:** Leverage ratios like debt ratio and debt to equity 
may not fully recover as companies took on permanent debt during COVID.
Profitability ratios like ROA and operating margin should show recovery.

In [0]:
%sql
SELECT
    period,
    ROUND(AVG(current_ratio), 4)    AS avg_current_ratio,
    ROUND(AVG(debt_ratio), 4)       AS avg_debt_ratio,
    ROUND(AVG(roa), 4)              AS avg_roa,
    ROUND(AVG(roe), 4)              AS avg_roe,
    ROUND(AVG(operating_margin), 4) AS avg_operating_margin,
    ROUND(AVG(gross_margin), 4)     AS avg_gross_margin,
    ROUND(AVG(asset_turnover), 4)   AS avg_asset_turnover,
    ROUND(AVG(debt_to_equity), 4)   AS avg_debt_to_equity
FROM edgar_kpi_clean
GROUP BY period
ORDER BY
    CASE period
        WHEN 'Pre-COVID' THEN 1
        WHEN 'COVID-Impact' THEN 2
        WHEN 'Post-COVID-Recovery' THEN 3
    END

## Q4 — Which sectors recovered fastest Post-COVID?
Compares ROA and operating margin by sector across all 3 periods.
Sectors with highest Post-COVID values recovered fastest.

In [0]:
%sql
WITH pre AS (
    SELECT sic,
        AVG(roa) AS avg_roa
    FROM edgar_kpi_clean
    WHERE period = 'Pre-COVID'
    AND sic IS NOT NULL
    GROUP BY sic
    HAVING COUNT(DISTINCT cik) >= 10
),
post AS (
    SELECT sic,
        AVG(roa) AS avg_roa
    FROM edgar_kpi_clean
    WHERE period = 'Post-COVID-Recovery'
    AND sic IS NOT NULL
    GROUP BY sic
    HAVING COUNT(DISTINCT cik) >= 10
)
SELECT
    pre.sic,
    ROUND(pre.avg_roa, 4)                AS pre_covid_roa,
    ROUND(post.avg_roa, 4)               AS post_covid_roa,
    ROUND(post.avg_roa - pre.avg_roa, 4) AS roa_delta
FROM pre
JOIN post ON pre.sic = post.sic
ORDER BY roa_delta DESC
LIMIT 22

## Q5 — Which KPIs showed most volatility during COVID-Impact?
Measures standard deviation of each KPI during COVID-Impact period.
High standard deviation = high volatility = unpredictable KPI behavior.

In [0]:
%sql
SELECT
    period,
    ROUND(STDDEV(current_ratio), 4)    AS stddev_current_ratio,
    ROUND(STDDEV(debt_ratio), 4)       AS stddev_debt_ratio,
    ROUND(STDDEV(roa), 4)              AS stddev_roa,
    ROUND(STDDEV(roe), 4)              AS stddev_roe,
    ROUND(STDDEV(operating_margin), 4) AS stddev_op_margin,
    ROUND(STDDEV(gross_margin), 4)     AS stddev_gross_margin,
    ROUND(STDDEV(asset_turnover), 4)   AS stddev_asset_turnover,
    ROUND(STDDEV(debt_to_equity), 4)   AS stddev_debt_to_equity
FROM edgar_kpi_clean
GROUP BY period
ORDER BY
    CASE period
        WHEN 'Pre-COVID' THEN 1
        WHEN 'COVID-Impact' THEN 2
        WHEN 'Post-COVID-Recovery' THEN 3
    END

## Q6 — How did leverage change during and after COVID?
Tracks debt ratio and debt to equity quarter by quarter across all periods.
High leverage during COVID indicates companies borrowed heavily to survive.
Recovery shown by declining debt ratio and debt to equity Post-COVID.

In [0]:
%sql
SELECT
    source_year,
    source_quarter,
    period,
    COUNT(DISTINCT cik)           AS companies,
    ROUND(AVG(debt_ratio), 4)     AS avg_debt_ratio,
    ROUND(AVG(debt_to_equity), 4) AS avg_debt_to_equity,
    ROUND(AVG(current_ratio), 4)  AS avg_current_ratio
FROM edgar_kpi_clean
GROUP BY period, source_year, source_quarter
ORDER BY source_year, source_quarter

## Q7 — Which companies improved vs deteriorated Pre to Post COVID?
Classifies companies as Improving, Stable or Deteriorating
based on ROA change from Pre-COVID to Post-COVID-Recovery.
Measures how many companies recovered financially after COVID.

In [0]:
%sql
WITH pre AS (
    SELECT cik,
        AVG(roa)              AS avg_roa,
        AVG(operating_margin) AS avg_op_margin
    FROM edgar_kpi_clean
    WHERE period = 'Pre-COVID'
    GROUP BY cik
),
post AS (
    SELECT cik,
        AVG(roa)              AS avg_roa,
        AVG(operating_margin) AS avg_op_margin
    FROM edgar_kpi_clean
    WHERE period = 'Post-COVID-Recovery'
    GROUP BY cik
)
SELECT
    CASE
        WHEN post.avg_roa > pre.avg_roa THEN 'Improving'
        WHEN post.avg_roa < pre.avg_roa THEN 'Deteriorating'
        ELSE 'Stable'
    END AS trend,
    COUNT(DISTINCT pre.cik)                               AS companies,
    ROUND(AVG(post.avg_roa - pre.avg_roa), 4)            AS avg_roa_change,
    ROUND(AVG(post.avg_op_margin - pre.avg_op_margin), 4) AS avg_op_margin_change
FROM pre
JOIN post ON pre.cik = post.cik
GROUP BY trend
ORDER BY companies DESC